In [ ]:
# Cell 0: synchronize the repository environment before running the workbench.
import subprocess
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    """Find the workspace root without confusing member pyproject files for it."""
    for candidate in (start, *start.parents):
        if (
            (candidate / "uv.lock").exists()
            and (candidate / "implementations").is_dir()
            and (candidate / "aieng-forecasting").is_dir()
        ):
            return candidate
    raise FileNotFoundError("Could not find the agentic-forecasting repository root.")


SETUP_ROOT = find_repo_root(Path.cwd().resolve())
print("Running: uv sync --all-extras --dev --all-packages")
setup_result = subprocess.run(
    ["uv", "sync", "--all-extras", "--dev", "--all-packages"],
    cwd=SETUP_ROOT,
    check=False,
    text=True,
    capture_output=True,
)
print(setup_result.stdout)
if setup_result.returncode != 0:
    print(setup_result.stderr)
    raise RuntimeError(
        "uv sync failed. Install uv or restart the notebook with the repository environment selected."
    )
print("Environment synchronized successfully.")
print("If packages changed, restart the notebook kernel before continuing.")


# Manufacturing Stress Forecasting Workbench

This notebook is the main interactive entry point for the manufacturing-stress experiment. It creates or refreshes the FRED data cache, runs the deterministic baselines, and optionally runs the token-limited LLMP backtest.

All predictors use the same backtest specification and shared Brier-score implementation. Complete LLMP results are cached by backtest-specification fingerprint so repeated analysis does not make new model calls. Historical agent prompts use relative month offsets instead of calendar dates, but this remains a retrospective pseudo-backtest: model-training knowledge leakage cannot be eliminated completely.


In [ ]:
# Main controls: change these values, then run the notebook from top to bottom.
REFRESH_FRED_DATA = False
BACKTEST_STRIDE = 3
RUN_LLMP_BACKTEST = False
FORCE_REFRESH_LLMP_CACHE = False
RUN_CURRENT_FORECAST = False

# BACKTEST_STRIDE = 1 evaluates every month; 3 evaluates every third month.
# False: one smoke run for historical frequency, logistic regression, and XGBoost.
# True: the same smoke run also includes the cached, token-limited LLMP.
# RUN_CURRENT_FORECAST controls the separate latest-data forecast section.
print({
    "refresh_fred_data": REFRESH_FRED_DATA,
    "backtest_stride": BACKTEST_STRIDE,
    "run_llmp_backtest": RUN_LLMP_BACKTEST,
    "force_refresh_llmp_cache": FORCE_REFRESH_LLMP_CACHE,
    "run_current_forecast": RUN_CURRENT_FORECAST,
})

## Imports and project paths

Run this notebook with the repository environment selected as the Jupyter kernel. The `.env` file is loaded for the FRED key and, when needed, the LLM proxy credentials.

In [ ]:
from pathlib import Path

import pandas as pd
import yaml
from aieng.forecasting.evaluation import BacktestSpec, backtest
from aieng.forecasting.methods import HistoricalFrequencyPredictor
from dotenv import load_dotenv
from manufacturing_stress_forecasting.analyst_agent import build_manufacturing_stress_agent_predictor
from manufacturing_stress_forecasting.data import IPMAN_SERIES_ID, build_manufacturing_stress_service
from manufacturing_stress_forecasting.predictors import (
    ManufacturingStressLogisticPredictor,
    ManufacturingStressXGBoostPredictor,
)
from manufacturing_stress_forecasting.run_agent_backtest import (
    run_or_load_backtest,
    validate_comparable_results,
)


REPO_ROOT = SETUP_ROOT
load_dotenv(REPO_ROOT / ".env", override=False)
SPEC_PATH = REPO_ROOT / "implementations" / "manufacturing_stress_forecasting" / "specs" / "manufacturing_stress_smoke.yaml"
STORE_DIR = REPO_ROOT / "data" / "predictions"
print(f"Repository: {REPO_ROOT}")


## Create or refresh the data

The service loads `IPMAN`, `DFF`, `DGS10`, and `DGS2` from the FRED cache. Set `REFRESH_FRED_DATA = True` in the first cell when the cache should be updated from FRED.

In [ ]:
service = build_manufacturing_stress_service(
    cache_dir=REPO_ROOT / "data" / "fred",
    refresh=REFRESH_FRED_DATA,
)
print(service.summary().to_string(index=False))

## Load the common backtest specification

Every predictor below receives this same target, horizon, forecast-origin schedule, warmup, and cutoff-aware data service.

In [ ]:
with SPEC_PATH.open() as file:
    spec = BacktestSpec.model_validate(yaml.safe_load(file))
spec = spec.model_copy(update={"stride": BACKTEST_STRIDE})

print(spec.task.description)
print(f"Origins: {spec.start.date()} to {spec.end.date()}, stride={spec.stride}, warmup={spec.warmup}")
print(f"Target: {spec.task.target_series_id}; horizon={spec.task.horizons[0]} {spec.task.frequency}")

## Run one smoke test

This is the single execution path. It always runs the historical-frequency, logistic-regression, and XGBoost predictors. Set `RUN_LLMP_BACKTEST = True` in the first cell to include the token-limited LLMP in the same run. Compatible complete results are loaded from cache unless `FORCE_REFRESH_LLMP_CACHE = True`; refreshed FRED data also forces a rerun.


In [ ]:
predictors = [
    HistoricalFrequencyPredictor(),
    ManufacturingStressLogisticPredictor(),
    ManufacturingStressXGBoostPredictor(),
]
if RUN_LLMP_BACKTEST:
    predictors.append(build_manufacturing_stress_agent_predictor(anonymize_dates=True))

results = {}
for predictor in predictors:
    if RUN_LLMP_BACKTEST:
        result = run_or_load_backtest(
            predictor,
            spec,
            service,
            force_refresh=FORCE_REFRESH_LLMP_CACHE or REFRESH_FRED_DATA,
            store_dir=STORE_DIR,
        )
    else:
        result = backtest(predictor=predictor, spec=spec, data_service=service)
    results[predictor.predictor_id] = result
    print(
        f"{result.predictor_id}: {result.mean_score:.4f} mean {result.metric}; "
        f"scored={len(result.scores)} skipped={result.skipped_origins}"
    )

validate_comparable_results(list(results.values()))
print("All predictors were scored on identical forecast origins.")


## Compare results

Lower Brier score is better. The notebook refuses to compare results unless every predictor was scored on the exact same forecast-origin and forecast-date pairs.


In [ ]:
comparison = pd.DataFrame(
    [
        {
            "predictor": result.predictor_id,
            "metric": result.metric,
            "mean_brier": result.mean_score,
            "scored": len(result.scores),
            "skipped": result.skipped_origins,
        }
        for result in results.values()
    ]
)
comparison.sort_values("mean_brier") if not comparison.empty else comparison

## Optional current forecast

Set `RUN_CURRENT_FORECAST = True` in the first cell to run the selected predictors against the latest released IPMAN data. This is a current forecast, not a historical backtest, so it does not produce a Brier score. `RUN_LLMP_BACKTEST` controls whether the LLMP is included here too.

In [ ]:
if RUN_CURRENT_FORECAST:
    with SPEC_PATH.open() as file:
        current_task = BacktestSpec.model_validate(yaml.safe_load(file)).task

    full_ipman = service.get_series(
        IPMAN_SERIES_ID,
        as_of=pd.Timestamp("2100-01-01").to_pydatetime(),
    )
    current_as_of = pd.Timestamp(full_ipman["released_at"].max())
    current_context = service.context(as_of=current_as_of.to_pydatetime())

    current_predictors = [
        HistoricalFrequencyPredictor(),
        ManufacturingStressLogisticPredictor(),
        ManufacturingStressXGBoostPredictor(),
    ]
    if RUN_LLMP_BACKTEST:
        current_predictors.append(build_manufacturing_stress_agent_predictor())

    print(f"Forecast origin: {current_as_of.date()}")
    print(
        "Latest visible IPMAN reference month: "
        f"{pd.Timestamp(current_context.get_series(IPMAN_SERIES_ID)['timestamp'].max()).date()}"
    )
    for predictor in current_predictors:
        prediction = predictor.predict(current_task, current_context)[0]
        print({
            "predictor": prediction.predictor_id,
            "forecast_date": str(pd.Timestamp(prediction.forecast_date).date()),
            "stress_probability": prediction.payload.probability,
            "metadata": prediction.metadata,
        })
else:
    print("Current forecast is disabled. Set RUN_CURRENT_FORECAST = True to enable it.")